# Research Paper Categorization Story

Given only paper title and abstract, can we predict category?

This notebook follows a clear narrative from raw text to model behavior and error analysis.


In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)
sns.set_theme(style='whitegrid')


## Load Dataset and Inspect Structure

Required columns:
- label: class name
- title: paper title
- abstract: paper abstract


In [ ]:
df = pd.read_csv('data/combined_papers_cleaned.csv')
use_cols = ['label', 'title', 'abstract']
df = df[use_cols].dropna().copy()
print('Shape:', df.shape)
display(df.head())


In [ ]:
sample_table = (
    df[['label', 'title']]
    .groupby('label', as_index=False)
    .head(1)
    .rename(columns={'label': 'category'})
)
display(sample_table)


## Class Distribution Visualization

Check imbalance before modeling.


In [ ]:
counts = df['label'].value_counts()
perc = (counts / counts.sum() * 100).round(2)
plot_df = pd.DataFrame({'category': counts.index, 'count': counts.values, 'pct': perc.values})

a = sns.barplot(data=plot_df, x='category', y='count', palette='Set2')
for i, r in plot_df.reset_index(drop=True).iterrows():
    a.text(i, r['count'], f"{int(r['count'])} ({r['pct']}%)", ha='center', va='bottom', fontsize=9)
plt.title('Class Distribution')
plt.show()


## Step-by-Step Text Preprocessing Pipeline

We show one abstract through: raw -> lower -> stopword-removed -> stemmed.


In [ ]:
stop_words = set(ENGLISH_STOP_WORDS)

def tokens(t):
    return re.findall(r'[a-z]+', str(t).lower())

def remove_stop(t):
    return ' '.join([w for w in tokens(t) if w not in stop_words])

def light_stem_word(w):
    for suf in ('ing', 'ed', 'ly', 'ies', 's'):
        if w.endswith(suf) and len(w) > len(suf) + 2:
            return (w[:-3] + 'y') if suf == 'ies' else w[:-len(suf)]
    return w

def light_stem(t):
    return ' '.join(light_stem_word(w) for w in tokens(t))

raw = df.iloc[0]['abstract']
lower = raw.lower()
no_stop = remove_stop(raw)
stemmed = light_stem(no_stop)

display(pd.DataFrame({
    'stage': ['raw', 'lowercased', 'stopwords_removed', 'lightly_stemmed'],
    'text': [raw, lower, no_stop, stemmed]
}))


In [ ]:
def top_terms(series, n=15):
    c = Counter()
    for t in series:
        c.update(tokens(t))
    return pd.DataFrame(c.most_common(n), columns=['term', 'count'])

before = top_terms(df['title'] + ' ' + df['abstract'])
after = top_terms((df['title'] + ' ' + df['abstract']).map(remove_stop).map(light_stem))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=before, x='count', y='term', ax=ax[0], color='#90caf9')
ax[0].set_title('Top terms before preprocessing')
sns.barplot(data=after, x='count', y='term', ax=ax[1], color='#80cbc4')
ax[1].set_title('Top terms after preprocessing')
plt.tight_layout(); plt.show()


## Combine Title and Abstract into One Feature

Title is short and high-signal; abstract adds context.


In [ ]:
df['combined_text'] = (df['title'].fillna('') + ' ' + df['abstract'].fillna('')).str.strip()
display(df[['title', 'abstract', 'combined_text']].head(2))


## TF, IDF, and TF-IDF Intuition with Toy Sentences

Formulas:

$$TF(t,d)=\frac{f_{t,d}}{\sum_j f_{j,d}}$$
$$IDF(t)=\log\left(\frac{N}{1+df(t)}\right)+1$$
$$TFIDF(t,d)=TF(t,d)\cdot IDF(t)$$


In [ ]:
toy_docs = [
    'graph neural networks for molecules',
    'neural networks for image classification',
    'bayesian inference for scientific models'
]
toy_vec = TfidfVectorizer()
toy_X = toy_vec.fit_transform(toy_docs)
display(pd.DataFrame(toy_X.toarray(), columns=toy_vec.get_feature_names_out()))


## Fit TF-IDF Vectorizer and Inspect Vocabulary


In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=12000, min_df=3, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['combined_text'])
feature_names = np.array(vectorizer.get_feature_names_out())
dfreq = np.asarray((X > 0).sum(axis=0)).ravel()

print('TF-IDF shape:', X.shape)
print('Vocabulary size:', len(feature_names))

most_idx = np.argsort(-dfreq)[:20]
least_idx = np.argsort(dfreq)[:20]
display(pd.DataFrame({'term': feature_names[most_idx], 'doc_freq': dfreq[most_idx]}))
display(pd.DataFrame({'term': feature_names[least_idx], 'doc_freq': dfreq[least_idx]}))


## TF-IDF Matrix Sparsity and Hyperparameter Effects


In [ ]:
sparsity = 1.0 - (X.nnz / (X.shape[0] * X.shape[1]))
print('Sparsity ratio:', round(sparsity, 4))

slice_mat = X[:10, :20].toarray()
plt.figure(figsize=(12, 4))
sns.heatmap(slice_mat, cmap='mako')
plt.title('TF-IDF heatmap slice (10 x 20)')
plt.show()

uni = TfidfVectorizer(stop_words='english', max_features=12000, min_df=3, ngram_range=(1,1)).fit(df['combined_text'])
bi = TfidfVectorizer(stop_words='english', max_features=12000, min_df=3, ngram_range=(1,2)).fit(df['combined_text'])
print('Unigram vocab size:', len(uni.get_feature_names_out()))
print('Unigram+Bigram vocab size:', len(bi.get_feature_names_out()))


## Implement Softmax and Cross-Entropy (From Scratch)


In [ ]:
def softmax(logits):
    z = logits - np.max(logits, axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(y_onehot, p, eps=1e-12):
    p = np.clip(p, eps, 1-eps)
    return -np.mean(np.sum(y_onehot * np.log(p), axis=1))

toy_logits = np.array([[2.0, 1.0, 0.1]])
toy_prob = softmax(toy_logits)
print('Softmax:', toy_prob.round(4), 'sum=', toy_prob.sum().round(4))

y_true = np.array([[1,0,0]])
print('Low loss:', round(cross_entropy(y_true, np.array([[0.9, 0.08, 0.02]])), 4))
print('High loss:', round(cross_entropy(y_true, np.array([[0.05, 0.9, 0.05]])), 4))


## Train/Test Split and Logistic Regression Training Loop

Do split before fitting to evaluate on unseen papers.


In [ ]:
label_names = sorted(df['label'].unique())
lab2i = {l:i for i,l in enumerate(label_names)}
i2lab = {i:l for l,i in lab2i.items()}

y = np.array([lab2i[x] for x in df['label']])
all_idx = np.arange(len(df))
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, all_idx, test_size=0.2, random_state=42, stratify=y
)

Xtr, Xte = X_train.toarray(), X_test.toarray()

k = len(label_names)
W = 0.01 * np.random.randn(Xtr.shape[1], k)
b = np.zeros((1, k))

def one_hot(y, k):
    o = np.zeros((len(y), k)); o[np.arange(len(y)), y] = 1; return o

Ytr = one_hot(y_train, k)
lr, l2, epochs = 0.5, 1e-4, 100
hist = {'loss': [], 'train_acc': [], 'test_acc': []}

for ep in range(epochs):
    logits = Xtr @ W + b
    p = softmax(logits)
    loss = cross_entropy(Ytr, p) + 0.5 * l2 * np.sum(W*W)

    dlog = (p - Ytr) / len(y_train)
    dW = Xtr.T @ dlog + l2 * W
    db = dlog.sum(axis=0, keepdims=True)

    W -= lr * dW
    b -= lr * db

    tr_pred = p.argmax(axis=1)
    te_pred = softmax(Xte @ W + b).argmax(axis=1)
    hist['loss'].append(loss)
    hist['train_acc'].append((tr_pred == y_train).mean())
    hist['test_acc'].append((te_pred == y_test).mean())

print('Learned weight matrix shape:', W.shape)


## Track Training Curves (Loss and Accuracy)


In [ ]:
e = np.arange(1, len(hist['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(e, hist['loss']); ax[0].set_title('Loss vs Epoch'); ax[0].set_xlabel('Epoch')
ax[1].plot(e, hist['train_acc'], label='Train'); ax[1].plot(e, hist['test_acc'], label='Test')
ax[1].set_title('Accuracy vs Epoch'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.tight_layout(); plt.show()


## Inspect Top Weighted Terms per Class


In [ ]:
fn = np.array(vectorizer.get_feature_names_out())
for cidx, cname in i2lab.items():
    top = np.argsort(W[:, cidx])[-12:]
    terms = fn[top]
    vals = W[top, cidx]
    plt.figure(figsize=(7,4))
    sns.barplot(x=vals, y=terms, orient='h', color='#a5d6a7')
    plt.title(f'Top weighted terms: {cname}')
    plt.tight_layout(); plt.show()


## Core Evaluation Metrics

Accuracy alone can mislead on imbalanced data, so we inspect richer metrics.


In [ ]:
probs = softmax(Xte @ W + b)
y_pred = probs.argmax(axis=1)
acc = accuracy_score(y_test, y_pred)
print('Overall accuracy:', round(acc, 4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix'); plt.show()

rep = classification_report(y_test, y_pred, target_names=label_names, output_dict=True)
display(pd.DataFrame(rep).T)


## Confidence Distribution and Calibration Check


In [ ]:
conf = probs.max(axis=1)
correct = (y_pred == y_test).astype(int)

sns.histplot(conf, bins=20, kde=True, color='#64b5f6')
plt.title('Prediction confidence (max softmax)')
plt.show()

grp = pd.DataFrame({'confidence': conf, 'correct': correct})
grp['bucket'] = pd.cut(grp['confidence'], bins=[0,0.4,0.6,0.8,1.0], include_lowest=True)
display(grp.groupby('bucket').agg(count=('correct','size'), accuracy=('correct','mean')).reset_index())


## Misclassification Review with Real Examples


In [ ]:
err = np.where(y_pred != y_test)[0][:5]
rows = []
for i in err:
    gi = idx_test[i]
    rows.append({
        'title': df.iloc[gi]['title'],
        'true_label': i2lab[y_test[i]],
        'pred_label': i2lab[y_pred[i]],
        'confidence': round(float(conf[i]), 4),
        'abstract_snippet': df.iloc[gi]['abstract'][:350] + '...'
    })
display(pd.DataFrame(rows))


## Conclusion

Our model classifies unseen papers from title + abstract with meaningful accuracy and interpretable term weights.

Potential improvements:
- richer preprocessing and phrase features
- more labeled data
- transformer-based text encoders


## Interactive Prediction Cell

Paste a new title + abstract below and get predicted class + confidence.


In [ ]:
new_title = 'A Survey of Efficient Transformer Compression Methods'
new_abstract = 'This survey reviews pruning, quantization, distillation, and low-rank adaptation methods for compressing large transformer models while preserving downstream performance.'

new_text = new_title + ' ' + new_abstract
new_x = vectorizer.transform([new_text]).toarray()
new_prob = softmax(new_x @ W + b)[0]
new_idx = int(np.argmax(new_prob))

print('Predicted category:', i2lab[new_idx])
print('Confidence:', round(float(new_prob[new_idx]), 4))

display(pd.DataFrame({'class': [i2lab[i] for i in range(k)], 'probability': new_prob}).sort_values('probability', ascending=False))
